In [ ]:
%pip install -q python-dotenv openai anthropic google-genai groq tiktoken

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
print("Keys loaded from .env")
print()

# ── The sentence used throughout ─────────────────────────────
SENTENCE = "What is a fixed deposit interest rate?"

print("Our sentence:")
print(f"  {SENTENCE!r}")
print(f"  characters : {len(SENTENCE)}")
print(f"  words      : {len(SENTENCE.split())}")
print()
print("Characters and words are NOT what the model counts.")
print("The model counts TOKENS. Let's find out how many.")
print()

# ════════════════════════════════════════════════════════════
print("=" * 60)
print("DEMO 1 — Step C: how many tokens? (ask each vendor)")
print("=" * 60)
print()
# ════════════════════════════════════════════════════════════



In [3]:
from openai    import OpenAI
from anthropic import Anthropic
from google    import genai
from groq      import Groq

openai_client    = OpenAI()
anthropic_client = Anthropic()
google_client    = genai.Client()
groq_client      = Groq()

print("Clients initialised")
print()

def count_openai(text):
    """Send to OpenAI and read prompt_tokens from the usage block."""
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": text}],
        max_tokens=5,           # tiny reply — we only want the usage count
    )
    return response.usage.prompt_tokens


def count_anthropic(text):
    """Use Anthropic's dedicated count_tokens endpoint — no model run at all."""
    result = anthropic_client.messages.count_tokens(
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": text}],
    )
    return result.input_tokens


def count_google(text):
    """Use Google's dedicated count_tokens endpoint — no model run at all."""
    result = google_client.models.count_tokens(
        model="gemini-2.5-flash",
        contents=text,
    )
    return result.total_tokens


def count_groq(text):
    """Send to Groq and read prompt_tokens from the usage block."""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": text}],
        max_tokens=5,
    )
    return response.usage.prompt_tokens

Clients initialised



In [ ]:
VENDOR_COUNTERS = {
    "OpenAI":    count_openai,
    "Anthropic": count_anthropic,
    "Google":    count_google,
    "Groq":      count_groq,
}

vendor_counts = {}   # name → token count, reused in Demo 2 and 3

print(f"Sentence : {SENTENCE!r}")
print()
print(f"{'VENDOR':12s}  {'TOKENS':6s}  VISUAL")
print("-" * 55)

for name, counter in VENDOR_COUNTERS.items():
    try:
        n = counter(SENTENCE)
        vendor_counts[name] = n
        bar = "█" * n
        print(f"  {name:12s}  {n:<6d}  {bar}")
    except Exception as e:
        print(f"  {name:12s}  skipped — {e}")

print()
print("The numbers are NOT all the same.")
print("Each vendor has its own tokenizer — its own way of splitting text.")
print("Same words, different rulers.")
print()


In [ ]:
import tiktoken

# NOTE: first run downloads a small vocabulary file then caches it.
# After that it works fully offline.

encoder     = tiktoken.get_encoding("o200k_base")   # tokenizer gpt-4o-mini uses
piece_ids   = encoder.encode(SENTENCE)
local_count = len(piece_ids)


print(f"tiktoken (local, o200k_base) says: {local_count} tokens")
print()
print(f"{'VENDOR':12s}  {'VENDOR COUNT':14s}  {'TIKTOKEN':10s}  DIFFERENCE")
print("-" * 55)

for name, vcount in vendor_counts.items():
    diff = local_count - vcount
    if diff == 0:
        sign = "exact match"
    elif diff > 0:
        sign = f"tiktoken over-counts by {diff}"
    else:
        sign = f"tiktoken under-counts by {abs(diff)}"
    print(f"  {name:12s}  {vcount:<14d}  {local_count:<10d}  {sign}")

print()
print("tiktoken matches OpenAI exactly — because tiktoken IS OpenAI's tokenizer.")
print("For other vendors it is close but not exact: a different ruler, same idea.")
print()




In [ ]:
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("DEMO 3 — Step D zoom: the actual pieces and integer IDs")
print("=" * 60)
print()
# ════════════════════════════════════════════════════════════

print(f"Sentence : {SENTENCE!r}")
print(f"Split into {local_count} pieces:\n")

print(f"  {'PIECE':4s}  {'TEXT':20s}  INTEGER ID")
print("  " + "-" * 42)

for i, piece_id in enumerate(piece_ids, start=1):
    piece_text = encoder.decode([piece_id])
    print(f"  {i:<4d}  {piece_text!r:<20s}  {piece_id}")

print()
print(f"The integer ID list the GPU actually receives:")
print(f"  {list(piece_ids)}")
print()
print("Each quoted piece is ONE token.")
print("Some are whole words, some are fragments, a leading space is")
print("part of the token — that is how sub-word tokenization works.")
print()

In [ ]:
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("DEMO 4 — Guard 3 simulation: pre-flight check before sending")
print("=" * 60)
print()
# ════════════════════════════════════════════════════════════

# Model context windows (approx, as of May 2026)
CONTEXT_WINDOWS = {
    "OpenAI  gpt-4o-mini":        128_000,
    "Anthropic claude-haiku-4-5": 200_000,
    "Google  gemini-2.5-flash":  1_000_000,
    "Groq    llama-3.3-70b":       32_768,
}

def preflight_check(text, model_label, context_window):
    """
    Simulate Guard 3 — count tokens locally and compare to context window.
    This is exactly what the data center does before the GPU runs.
    """
    count = len(encoder.encode(text))
    fits  = count <= context_window
    status = "PASS — safe to send" if fits else "FAIL — will get 400"
    margin = context_window - count
    print(f"  {model_label}")
    print(f"    token count    : {count:>10,}")
    print(f"    context window : {context_window:>10,}")
    print(f"    headroom       : {margin:>10,} tokens remaining")
    print(f"    Guard 3 result : {status}")
    print()
    return fits


print(f"Checking: {SENTENCE!r}")
print()

for label, window in CONTEXT_WINDOWS.items():
    preflight_check(SENTENCE, label, window)

print("All pass for our short sentence — as expected.")
print()
print("Now let's try a deliberately oversized payload (100K 'banking' words):")
print()

big_text = "banking " * 100_000   # ~100K tokens

for label, window in CONTEXT_WINDOWS.items():
    preflight_check(big_text, label, window)

print("Groq fails — Llama's 32K context window is the smallest of the four.")
print("This is exactly the 400 your sweep produced earlier.")